# Part 2: Tuned Logistic Regression Baseline
Training was executed by `src/baseline.py`. This notebook displays its saved results and provenance; it does not retrain models. All evaluation uses development data only. The sealed test set is not opened.

Reproduce training from the project root:
```bash
.venv/bin/python src/baseline.py
.venv/bin/python src/verify_baseline.py
```

In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
out = root / 'reports/baseline'
summary = json.loads((out / 'summary.json').read_text())
provenance = json.loads((out / 'provenance.json').read_text())
verification = json.loads((out / 'verification.json').read_text())
assert summary['status'] == verification['status'] == 'SUCCESS'
assert summary['sealed_test_accessed'] is False
for name in ['src/baseline.py', 'src/pipeline.py', 'config/baseline_protocol.json']:
    assert hashlib.sha256((root / name).read_bytes()).hexdigest() == provenance['input_files_sha256'][name]
print('Training runtime (seconds):', summary['runtime_seconds'])
print('Total pipeline fits:', summary['total_pipeline_fits'])
print('Convergence failures:', summary['convergence_failures'])
print('Transaction files read:', provenance['transaction_files_read'])
print('Audit:', verification['status'])

Training runtime (seconds): 49.55690725008026
Total pipeline fits: 186
Convergence failures: 0
Transaction files read: ['data/processed/development.csv']
Audit: SUCCESS


## Outer validation results
Each outer validation fold was excluded from scaling, model fitting, C selection and threshold selection for that fold. Inner five-fold CV chooses C by mean AP. For the selected C, inner OOF predictions select the threshold minimizing `(FP + 50*FN) / N`.

The default 0.5 threshold is reported separately from the cost-tuned threshold. AP uses probabilities and is identical for both decisions.

In [2]:
folds = pd.read_csv(out / 'outer_fold_results.csv')
columns = ['outer_fold', 'validation_rows', 'validation_frauds', 'selected_C',
           'AP', 'default_FP', 'default_FN', 'default_mean_cost',
           'tuned_threshold', 'tuned_FP', 'tuned_FN', 'tuned_mean_cost']
display(folds[columns])
display(pd.DataFrame(summary['metrics']).T)
print('Pooled confusion counts:', summary['pooled_confusion'])

,outer_fold,validation_rows,validation_frauds,selected_C,AP,default_FP,default_FN,default_mean_cost,tuned_threshold,tuned_FP,tuned_FN,tuned_mean_cost
0,1,44106,75,100.000,0.711697,7,32,0.036435,0.015371,72,13,0.016370
1,2,44106,75,0.100,0.776384,6,28,0.031878,0.016102,49,11,0.013581
2,3,44106,76,0.010,0.682390,7,35,0.039836,0.012561,57,16,0.019430
3,4,44106,76,0.001,0.738381,9,38,0.043282,0.005027,59,11,0.013808
4,5,44106,76,0.001,0.799652,4,37,0.042035,0.004395,95,9,0.012357


,outer_fold_mean,outer_fold_sample_SD
AP,0.741701,0.047409
default_mean_cost,0.038693,0.004612
tuned_mean_cost,0.015109,0.002823
default_precision,0.863289,0.037268
default_recall,0.550526,0.050963
tuned_precision,0.495931,0.059019
tuned_recall,0.841263,0.034865


Pooled confusion counts: {'default': {'TN': 220119, 'FP': 33, 'FN': 170, 'TP': 208}, 'tuned': {'TN': 219820, 'FP': 332, 'FN': 60, 'TP': 318}}


## Full-development selection and limitations
Full-development tuning selects the artifact for later evaluation. Its inner score is a tuning score, not independent held-out performance. The primary performance estimate is mean outer-fold AP. Pooled OOF AP is an additional summary and can differ because AP is nonlinear and the folds use different fitted models.

Fold SD describes variability; it is not a confidence interval. Formal error bars, class-weight comparisons and the final submission are not completed by this part.

In [3]:
grid = pd.read_csv(out / 'full_development_inner_scores.csv')
display(grid.groupby('C')['AP'].agg(['mean', 'std']))
print('Full-development selection:', summary['full_development_selection'])
print('Primary mean outer-fold AP:', summary['metrics']['AP']['outer_fold_mean'])
print('Additional pooled OOF AP:', summary['pooled_outer_oof_AP'])
print('Error bar status:', summary['error_bar_status'])

,mean,std
C,,
0.001,0.745166,0.051489
0.010,0.744416,0.050722
0.100,0.748307,0.046920
1.000,0.749654,0.044148
10.000,0.749203,0.043886
100.000,0.749397,0.043941


Full-development selection: {'C': 1.0, 'inner_mean_AP': 0.7496543347067656, 'threshold': 0.014549755567758683, 'inner_threshold_selection_cost': 0.014428875889901601}
Primary mean outer-fold AP: 0.7417010013413197
Additional pooled OOF AP: 0.7318366826255067
Error bar status: not_yet_computed; fold SD is descriptive spread, not a confidence interval
